# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Accessed as an object, not as a dictionary
metadata_json = dataset.metadata.to_json()
print(f"{metadata_json['name']}: {metadata_json['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Here we'll list the `@id` for each record set, the fields it contains (by `@id`), and the available columns.

In [ ]:
# Gather all available record set @ids
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    field_ids = [f.id for f in rs.fields]
    print(f"  Field @ids: {field_ids}")
    col_ids = [f.column.id if f.column else None for f in rs.fields]
    print(f"  Column @ids: {col_ids}")

Let's preview a few records from the main record set. Replace `<main_record_set_id>` with the `@id` of the primary record set you want to browse (as listed above).

In [ ]:
# Let's view the first few records from the primary record set
main_record_set_id = dataset.record_sets[0].id  # e.g., 'cr:RecordSet/mental_health_records'
print(f"Previewing records from record set: {main_record_set_id}")
for idx, record in enumerate(dataset.records(record_set=main_record_set_id)):
    print(record)
    if idx>=2:
        break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. We use the record set `@id`s as discovered above.

In [ ]:
# Extract data from all record sets
all_record_sets = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rs_id in all_record_sets:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded record set {rs_id} with {len(df)} rows and columns: {list(df.columns)}\n")
    else:
        print(f"Record set {rs_id} has no records (empty DataFrame).\n")

# Pick the main record set for further EDA, e.g., first non-empty one
for rs_id, df in dataframes.items():
    main_df = df
    main_rs_id = rs_id
    break

print(f"\nMain record set '{main_rs_id}' DataFrame preview and columns:")
display(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming distributions, or grouping by attributes as needed.

We'll:
- Select a numeric field by its `@id` (e.g. `'cr:field/phq9_total'` for PHQ-9 total score, adjust as appropriate for your dataset)
- Filter by value
- Normalize and group

In [ ]:
# The numeric field and group field @ids must match your actual dataset fields.
# We'll list possible numeric fields based on columns in main_df
print("Numeric-like fields in main_df:")
print([col for col in main_df.columns if main_df[col].dtype in ['int64','float64','int32','float32'] or pd.api.types.is_numeric_dtype(main_df[col])])

# Example: Let's assume 'cr:field/phq9_total' is the @id for PHQ-9 total score and 'cr:field/gender' for gender
numeric_field_id = None
group_field_id = None
for col in main_df.columns:
    if 'phq' in col.lower() and ('total' in col.lower() or 'score' in col.lower()):
        numeric_field_id = col
    if 'gender' in col.lower():
        group_field_id = col

if numeric_field_id is None:
    numeric_field_id = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])][0]  # as fallback
if group_field_id is None:
    group_field_id = main_df.columns[0]  # fallback

print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field: {group_field_id}")

threshold = main_df[numeric_field_id].dropna().quantile(0.8)  # filter top 20% (as example)
filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, norm_col]].head())

if group_field_id in main_df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id} (showing means):")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Below we plot the distribution of the numeric field and show means by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Plot histogram of the numeric field
plt.figure(figsize=(8,5))
sns.histplot(main_df[numeric_field_id].dropna(), kde=True, bins=15)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot by group if group field is categorical
if group_field_id in main_df.columns:
    plt.figure(figsize=(7,5))
    sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded and explored the dataset using its Croissant schema and the `mlcroissant` library
- Identified and extracted the main record set and fields using their `@id`s
- Performed EDA including filtering, normalization, grouping, and visualization
- This process provides a foundation for further in-depth statistical analysis, modeling, or reporting using FAIR-certified survey data.

> **Next Steps:**
> - Explore more advanced analyses, such as correlations between different psychological scores
> - Integrate with external health or demographic datasets for further research
> - Share findings and reusable code notebooks with the community!